# 13 Service Efficiency Analysis

## Purpose

This notebook extracts and analyzes METRO Houston route productivity data from 12 monthly ridership report PDFs covering October 2024 through September 2025.

The goal is to move beyond raw ridership totals and evaluate how efficiently each route generates ridership relative to the amount of service provided.

## Research Question

Which METRO routes generate the most ridership per unit of service?

## Inputs

- Monthly METRO ridership report PDFs stored in `data/raw/metro_ridership/`
- Local route productivity tables from pages 3 and 4 of each report

## Outputs

- `data/processed/service_efficiency_all_months.csv`
- `data/processed/route_efficiency_summary.csv`

## Why This Matters

A route with high ridership is not automatically efficient. Some routes carry many riders because they receive a large amount of service, while others may carry fewer total riders but generate more riders per revenue hour or revenue mile.

This notebook creates the route productivity dataset used later for investment-priority, accessibility, and ROI-style analysis.


## 1. Import Libraries

This notebook uses:

- `pandas` for tabular data processing
- `tabula-py` for extracting tables from PDFs
- `pathlib` for file path handling
- `re` for extracting month labels from filenames
- `matplotlib` for basic visualization


In [ ]:
import pandas as pd
from pathlib import Path
import tabula
import re
import matplotlib.pyplot as plt

## 2. Locate Monthly Ridership Reports

All monthly ridership PDFs should be stored in:

`data/raw/metro_ridership/`

The notebook finds every PDF in that folder and stores them in a sorted list so the same extraction process can be applied to all 12 months.


In [ ]:
pdf_dir = Path("../data/raw/metro_ridership")

pdf_files = sorted(pdf_dir.glob("*.pdf"))

print(f"{len(pdf_files)} PDFs found")

pdf_files

## 3. Extract Local Route Tables from Each PDF

METRO's local route productivity tables are split across two pages in the ridership reports.

- Page 3 contains the first portion of the local route network
- Page 4 contains the remaining local routes and some related services

Both pages are extracted from every PDF. The extracted tables are stored temporarily before cleaning.


In [ ]:
all_tables = []

for pdf in pdf_files:

    print(f"Reading {pdf.name}")

    for page in [3, 4]:

        try:
            tables = tabula.read_pdf(
                str(pdf),
                pages=page,
                multiple_tables=True,
                lattice=True,
                force_subprocess=True
            )

            if len(tables) > 0:

                table = tables[0].copy()
                table["source_pdf"] = pdf.stem
                table["page"] = page

                all_tables.append(table)

                print(f"  page {page}: success")

            else:
                print(f"  page {page}: no tables found")

        except Exception as e:
            print(f"  page {page}: failed")
            print(e)

print()
print(f"Collected {len(all_tables)} tables")

## 4. Define Cleaning Functions

The PDF tables are not perfectly consistent across months.

These helper functions:

- Extract the report month from the PDF filename
- Remove extra metadata columns
- Remove repeated header rows
- Standardize column names
- Convert ridership and productivity fields to numeric values

This creates a consistent route-month dataset across all reports.


In [ ]:
def extract_month_from_filename(source_pdf):
    match = re.search(
        r"(October|November|December|January|February|March|April|May|June|July|August|September)\s+\d{4}",
        source_pdf
    )

    if match:
        return match.group(0)

    return source_pdf


def clean_monthly_table(table, source_pdf):
    df = table.copy()

    month = extract_month_from_filename(source_pdf)

    for col in ["source_pdf", "page"]:
        if col in df.columns:
            df = df.drop(columns=[col])

    first_cell = str(df.iloc[0, 0]).strip()

    if first_cell == "#":
        data_start = 1
    else:
        data_start = 2

    cleaned = df.iloc[data_start:].copy()

    cleaned = cleaned.iloc[:, 0:11]

    cleaned.columns = [
        "route_id",
        "route_name",
        "weekday_avg_boardings",
        "weekday_boardings_per_revenue_hour",
        "weekday_boardings_per_revenue_mile",
        "saturday_avg_boardings",
        "saturday_boardings_per_revenue_hour",
        "saturday_boardings_per_revenue_mile",
        "sunday_avg_boardings",
        "sunday_boardings_per_revenue_hour",
        "sunday_boardings_per_revenue_mile",
    ]

    cleaned["month"] = month
    cleaned["source_pdf"] = source_pdf

    cleaned = cleaned[
        cleaned["route_id"].notna()
    ]

    cleaned = cleaned[
        cleaned["route_id"].astype(str).str.strip().str.isnumeric()
    ]

    numeric_cols = [
        "route_id",
        "weekday_avg_boardings",
        "weekday_boardings_per_revenue_hour",
        "weekday_boardings_per_revenue_mile",
        "saturday_avg_boardings",
        "saturday_boardings_per_revenue_hour",
        "saturday_boardings_per_revenue_mile",
        "sunday_avg_boardings",
        "sunday_boardings_per_revenue_hour",
        "sunday_boardings_per_revenue_mile",
    ]

    for col in numeric_cols:
        cleaned[col] = (
            cleaned[col]
            .astype(str)
            .str.replace(",", "", regex=False)
        )

        cleaned[col] = pd.to_numeric(
            cleaned[col],
            errors="coerce"
        )

    return cleaned

## 5. Clean Extracted Tables

Each extracted table is passed through the cleaning function.

The result is a list of standardized tables with the same columns and numeric formats.


In [ ]:
cleaned_tables = []

for table in all_tables:

    source_pdf = table["source_pdf"].iloc[0]

    cleaned = clean_monthly_table(
        table,
        source_pdf
    )

    cleaned_tables.append(cleaned)

print(f"Cleaned {len(cleaned_tables)} tables")

## 6. Combine Monthly Tables

The cleaned tables are combined into one long-format dataset.

Each row represents one route in one monthly ridership report.


In [ ]:
service_efficiency = pd.concat(
    cleaned_tables,
    ignore_index=True
)

print(service_efficiency.shape)

service_efficiency.head()

## 7. Validate Monthly Coverage

This check counts the number of extracted route records by month.

Most months should contain a similar number of routes. Small differences may occur because PDF formatting can vary slightly from month to month.


In [ ]:
service_efficiency["month"].value_counts()

## 8. Inspect Extracted Data

This step verifies that the cleaned dataset contains the expected route IDs, route names, ridership metrics, productivity metrics, month labels, and source PDF names.


In [ ]:
service_efficiency.tail()

## 9. Save Monthly Service Efficiency Dataset

The long-format route-month dataset is saved for reuse in later notebooks.

This file preserves monthly variation across the full 12-month period.


In [ ]:
service_efficiency.to_csv(
    "../data/processed/service_efficiency_all_months.csv",
    index=False
)

print("saved service_efficiency_all_months.csv")

## 10. Create Route-Level Annual Efficiency Summary

Monthly route observations are aggregated into annual averages.

For each route, this summary calculates:

- Average weekday boardings
- Average boardings per revenue hour
- Average boardings per revenue mile
- Average Saturday boardings
- Average Sunday boardings
- Number of months observed

This summary is easier to use for route ranking and investment analysis.


In [ ]:
route_efficiency_summary = (
    service_efficiency
    .groupby(
        ["route_id", "route_name"],
        as_index=False
    )
    .agg(
        avg_weekday_boardings=(
            "weekday_avg_boardings",
            "mean"
        ),
        avg_boardings_per_revenue_hour=(
            "weekday_boardings_per_revenue_hour",
            "mean"
        ),
        avg_boardings_per_revenue_mile=(
            "weekday_boardings_per_revenue_mile",
            "mean"
        ),
        avg_saturday_boardings=(
            "saturday_avg_boardings",
            "mean"
        ),
        avg_sunday_boardings=(
            "sunday_avg_boardings",
            "mean"
        ),
        months_observed=(
            "month",
            "nunique"
        )
    )
)

print(route_efficiency_summary.shape)

route_efficiency_summary.head()

## 11. Calculate Productivity Score

A productivity score is created by combining:

- Boardings per revenue hour
- Boardings per revenue mile

This rewards routes that perform well relative to both service time and service distance.


In [ ]:
route_efficiency_summary["productivity_score"] = (
    route_efficiency_summary["avg_boardings_per_revenue_hour"]
    *
    route_efficiency_summary["avg_boardings_per_revenue_mile"]
)

route_efficiency_summary[
    [
        "route_name",
        "avg_weekday_boardings",
        "avg_boardings_per_revenue_hour",
        "avg_boardings_per_revenue_mile",
        "productivity_score"
    ]
].sort_values(
    "productivity_score",
    ascending=False
).head(20)

## 12. Visualize Top Productivity Routes

This chart highlights the highest-performing routes based on the composite productivity score.


In [ ]:
top15_productivity = route_efficiency_summary.sort_values(
    "productivity_score",
    ascending=False
).head(15)

plt.figure(figsize=(10, 6))

plt.barh(
    top15_productivity["route_name"],
    top15_productivity["productivity_score"]
)

plt.gca().invert_yaxis()

plt.title("Top 15 METRO Routes by Productivity Score")
plt.xlabel("Productivity Score")
plt.ylabel("Route")

plt.tight_layout()
plt.show()

## 13. Save Route-Level Efficiency Summary

The route-level productivity summary is saved for use in later notebooks, including:

- Bus and rail context analysis
- Accessibility analysis
- Transit investment ROI analysis
- Final recommendations


In [ ]:
route_efficiency_summary.to_csv(
    "../data/processed/route_efficiency_summary.csv",
    index=False
)

print("saved route_efficiency_summary.csv")

## Summary

This notebook automated the extraction and processing of route productivity data from 12 monthly METRO Houston ridership reports.

Key outputs:

- `service_efficiency_all_months.csv`
- `route_efficiency_summary.csv`

The final dataset provides route-level measures of:

- Average weekday ridership
- Boardings per revenue hour
- Boardings per revenue mile
- Weekend ridership
- Composite productivity score

These outputs support later analysis of transit productivity, accessibility, investment prioritization, and future high-capacity transit opportunities.

## Data Quality Note

The extracted route count varies slightly across months because the source PDFs are formatted inconsistently. The `months_observed` field is included so later notebooks can identify routes with incomplete monthly coverage.
